In [ ]:
import gzip
import json
import networkx
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
import seaborn as sns
import os

from matplotlib.backends.backend_pdf import PdfPages

from utils.io import load_analyzer, load_clustering, get_review_pmids
from utils.preprocessing import preprocess_clustering, get_clustering_level

sns.set();
%matplotlib inline

## Clustering Overview

### Visualize ground truth clustering and their intersections as a contingency matrix.

In [ ]:
CLUSTERING_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/clustering/'

In [ ]:
with PdfPages('clustering-overview.pdf') as pdf:
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level)
            n_clusters = len(processed_clustering)
            intersections = np.zeros((n_clusters, n_clusters))
            for i in range(n_clusters):
                for j in range(n_clusters):
                    common_elements = set(processed_clustering[i]).intersection(processed_clustering[j])
                    intersections[i, j] = len(common_elements)

            plt.figure(figsize = (10,7))
            sns.heatmap(intersections, annot=True)
            plt.title(f'{filename} - LEVEL {level}')
            pdf.savefig()
            plt.close()

### Ground Truth Cluster Size and Similarity Graph Density

In [ ]:
MIN_COCITATION = 2
MIN_BIBCOUPLING = 0

In [ ]:
def get_param_subgraph(graph, param, min_value=None):
    """
    Get a subgraph of a similarity graph that contains only edges with `param`.
    Optionally minimum value of `param` can be specified.
    """
    param_edges = list(filter(lambda el: param in el[2] and (not min_value or el[2][param] >= min_value), 
                              graph.edges(data=True)))
    param_subgraph = graph.edge_subgraph([(s, e) for s, e, d in param_edges])
    return param_subgraph

In [ ]:
size_density_data = []

for pmid in get_review_pmids():
    analyzer = load_analyzer(pmid)
    clustering = load_clustering(pmid)
    for level in range(1, get_clustering_level(clustering)):
        ground_truth = preprocess_clustering(clustering, max_level=level,
                                             include_box_sections=False,
                                             uniqueness_method='unique_only')
        ground_truth_clusters = {comp: [] for comp in ground_truth.values()}
        for ref, c in ground_truth.items():
            ground_truth_clusters[c].append(ref)
        for c, refs in ground_truth_clusters.items():
            cluster_similarity_subgraph = analyzer.similarity_graph.subgraph(refs)
            
            param_densities = []
            for param, min_value in zip(['cocitation', 'cocitation', 'bibcoupling', 'citation', 'text'], [0, 2, 0, 0, 0]):
                param_subgraph = get_param_subgraph(cluster_similarity_subgraph, param, min_value)
                param_densities.append(networkx.density(param_subgraph))
            size_density_data.append((pmid, level, 
                                      len(refs) / len(ground_truth), 
                                      networkx.density(cluster_similarity_subgraph),
                                      *param_densities))

In [ ]:
size_density_df = pd.DataFrame(size_density_data, 
                               columns=['PMID', 'Level', 'Relative Size',
                                        'Overall Density', 'Cocitation Density', 
                                        'Cocitation (min=2) Density', 'Bibcoupling Density',
                                        'Citation Density', 'Text Citation Density']).apply(pd.to_numeric, errors='coerce')

In [ ]:
size_density_df.head(5)

In [ ]:
size_density_df.max()

#### Relative Cluster Size vs Densities of Subgraphs for Different Parameters

In [ ]:
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6)) = plt.subplots(3, 2, figsize=(16, 24))

for ax, param in zip([ax1, ax2, ax3, ax4, ax5, ax6], 
                     ['Overall Density', 'Cocitation Density', 
                      'Cocitation (min=2) Density', 'Bibcoupling Density', 
                      'Citation Density', 'Text Citation Density']):
    sc = ax.scatter(x=size_density_df["Relative Size"], y=size_density_df[param], 
                    c=size_density_df["Level"], cmap='jet')
    ax.set_xlabel('Relative Size')
    ax.set_ylabel('Density');
    ax.set_title(param)

#### Correlation of Subgraph Density for Different Parameters

In [ ]:
pd.plotting.scatter_matrix(size_density_df[['Cocitation (min=2) Density', 'Bibcoupling Density', 
                                            'Citation Density', 'Text Citation Density']],
                           figsize=(16, 16));

### References Subgraph Density (with min=2 for Cocitation, otherwise does not make sense)

In [ ]:
refs_density_data = []
overall_density_data = []

for pmid in get_review_pmids():
    analyzer = load_analyzer(pmid)
    overall_density_data.append(networkx.density(analyzer.similarity_graph))
    
    clustering = load_clustering(pmid)
    for level in range(1, get_clustering_level(clustering)):
        ground_truth = preprocess_clustering(clustering, max_level=level,
                                             include_box_sections=False,
                                             uniqueness_method='unique_only')
        references_subgraph = analyzer.similarity_graph.subgraph(ground_truth.keys())
        
        # Filter out edges with cocitation = 1
        good_edges = list(filter(lambda el: 'cocitation' not in el[2] or el[2]['cocitation'] >= 2, 
                                 references_subgraph.edges(data=True)))
        good_subgraph = references_subgraph.edge_subgraph([(s, e) for s, e, d in good_edges])
        
        n_references = good_subgraph.number_of_nodes()
        refs_density_data.append(networkx.density(good_subgraph))
#         print(pmid, level, n_references * (n_references - 1) / 2, good_subgraph.number_of_edges())

In [ ]:
plt.figure(figsize=(8, 6))
bins = np.linspace(0.0, 1.2, 25)
plt.hist(overall_density_data, bins, color='blue', alpha=0.5, label='All papers')
plt.hist(refs_density_data, bins, color='green', alpha=0.5, label='Only references')
plt.xlabel('Density')
plt.ylabel('Number of clusterings')
plt.legend();

## PubTrends Export Overview

### Prior vs. Derivative Papers

In [ ]:
year_diffs = []

for pmid in get_review_pmids():
    analyzer = load_analyzer(pmid)
    review_row = analyzer.df[analyzer.df.id == pmid]
    review_year = review_row['year'].values[0]
    ref_year_diffs = analyzer.df['year'].drop(review_row.index) - review_year
    year_diffs.extend(list(ref_year_diffs.values))

In [ ]:
max(year_diffs)

In [ ]:
plt.hist(year_diffs, bins=100)
plt.xlim([-50, 5])
plt.xlabel('Year Relative to the Review Paper')
plt.ylabel('Amount of Papers')

### Direct Citations Count

In [ ]:
total_review_citations = 0
total_review_total = 0

for pmid in get_review_pmids():
    analyzer = load_analyzer(pmid)
    review_row = analyzer.df[analyzer.df.id == pmid]
    review_citations = len(list(analyzer.citations_graph.predecessors(pmid)))
    review_total = review_row['total'].values[0]
    total_review_citations += review_citations
    total_review_total += review_total
    
print('Review citations included in the analysis:', total_review_citations)
print('Total citation count of all reviews:', total_review_total)

### Edge Count and Weight Distribution

In [ ]:
def rebuild_similarity_graph(analyzer):
    analyzer.ids = set(analyzer.df['id'])
    analyzer.n_papers = len(analyzer.ids)
    analyzer.pub_types = list(set(analyzer.df['type']))
    analyzer.query = 'restored from PubTrends export'
    
    PUB_DF_COLUMNS = ['id', 'title', 'abstract', 'year', 'type', 'keywords', 'mesh', 'doi', 'aux']
    analyzer.pub_df = analyzer.df[PUB_DF_COLUMNS]
    
    analyzer.components = set(analyzer.df['comp'].unique())
    if -1 in analyzer.components:
        analyzer.components.remove(-1)
        
    settings = AnalyzerSettings()
    analyzer.corpus_ngrams, analyzer.corpus_counts = \
        vectorize_corpus(analyzer.pub_df,
                         max_features=settings.VECTOR_WORDS, n_gram=settings.VECTOR_NGRAMS,
                         min_df=settings.VECTOR_MIN_DF, max_df=settings.VECTOR_MAX_DF)
    tfidf = compute_tfidf(analyzer.corpus_counts)
    analyzer.texts_similarity = analyzer.analyze_texts_similarity(analyzer.pub_df, tfidf,
                                                                  settings.SIMILARITY_TEXT_MIN,
                                                                  settings.SIMILARITY_TEXT_CITATION_N)
    
    cocit_data = []
    bibcoupling_data = []
    for cited_1, cited_2, data in analyzer.similarity_graph.edges(data=True):
        if 'cocitation' in data:
            cocit_data.append((cited_1, cited_2, data['cocitation']))
        if 'bibcoupling' in data:
            bibcoupling_data.append((cited_1, cited_2, data['bibcoupling']))
            
    cocit_df = pd.DataFrame(cocit_data, columns=['cited_1', 'cited_2', 'total'])
    bibcoupling_df = pd.DataFrame(bibcoupling_data, columns=['citing_1', 'citing_2', 'total'])
    
    analyzer.similarity_graph = analyzer.build_similarity_graph(
        analyzer.df, analyzer.texts_similarity,
        analyzer.citations_graph, cocit_df, bibcoupling_df,
        current=0, task=None
    )

In [ ]:
with PdfPages('pubtrends-similarity-parameters-scaled-hist.pdf') as pdf:
    for filename in sorted(os.listdir(PUBTRENDS_EXPORT_FOLDER)):
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, filename)
        analyzer = reload_exported_analyzer(analysis_file)
        rebuild_similarity_graph(analyzer)
        stats = {
            'cocitation': [],
            'bibcoupling': [],
            'citation': [],
            'text': []
        }
        for _, _, data in analyzer.similarity_graph.edges(data=True):
            for key in stats.keys():
                stats[key].append(data.get(key, 0))

        stats_df = pd.DataFrame(stats).apply(pd.to_numeric, errors='coerce')
        ax = stats_df.hist(log=True)
        plt.suptitle(filename)
        pdf.savefig()
        plt.close()

In [ ]:
full_stats = {
    'cocitation': [],
    'bibcoupling': [],
    'citation': [],
    'text': []
}

from utils.analysis import reload_exported_analyzer

for filename in sorted(os.listdir(PUBTRENDS_EXPORT_FOLDER)):
    analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, filename)
    analyzer = reload_exported_analyzer(analysis_file)
    
    stats = {
        'cocitation': 0,
        'bibcoupling': 0,
        'citation': 0,
        'text': 0
    }
    
    for _, _, data in analyzer.similarity_graph.edges(data=True):
        for key in stats.keys():
            if key in data:
                stats[key] += 1
    
    for key in stats:
        full_stats[key].append(stats[key])
    
    print(filename, analyzer.similarity_graph.number_of_edges(), stats)

In [ ]:
full_stats_df = pd.DataFrame(full_stats).apply(pd.to_numeric, errors='coerce')

In [ ]:
full_stats_df.hist()

### Find Papers That Are Most Cited by Review References

In [ ]:
from collections import Counter

most_cited_ref_ref_data = []

for pmid in get_review_pmids():
    analyzer = load_analyzer(pmid)
    citations_graph = analyzer.citations_graph
    ref_refs = []
    for ref in citations_graph.successors(pmid):
        ref_refs.extend(list(citations_graph.successors(ref)))
    for ref_ref_pmid, count in Counter(ref_refs).most_common(5):
        most_cited_ref_ref_data.append((
            pmid, ref_ref_pmid, count, 
            float(count) / analyzer.df[analyzer.df.id == ref_ref_pmid]['total'].values[0]
        ))

In [ ]:
most_cited_ref_ref_df = pd.DataFrame(most_cited_ref_ref_data,
                                     columns=['Review PMID', "Refs' Ref PMID", "Absolute Count", "Relative Count"])

In [ ]:
most_cited_ref_ref_df.sort_values(by='Relative Count', ascending=False).head(10)

## DEPRECATED: Estimate PubTrends quality for different clusterings

In [ ]:
from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig, AnalyzerSettings
from pysrc.papers.db.loaders import Loaders
from pysrc.papers.utils import vectorize_corpus, compute_tfidf

from sklearn.metrics.cluster import adjusted_rand_score, v_measure_score, \
                                    adjusted_mutual_info_score, contingency_matrix
from sklearn.metrics import fowlkes_mallows_score

In [ ]:
PUBTRENDS_EXPORT_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/pubtrends_export/'
PUBTRENDS_CONFIG = PubtrendsConfig(test=False)

In [ ]:
def get_clustering(analyzer, target_ids):
    data = analyzer.df[analyzer.df['id'].isin(target_ids)]
    return {k: v for k, v in zip(data['id'], data['comp'])}

In [ ]:
def evaluate_clustering(analyzer, ground_truth, metric):
    actual_clustering = get_clustering(analyzer, ground_truth.keys())
    
    # Align clusterings
    labels_true = []
    labels_pred = []

    for pmid in actual_clustering:
        labels_true.append(ground_truth[pmid])
        labels_pred.append(actual_clustering[pmid])
        
    return metric(labels_true, labels_pred), contingency_matrix(labels_true, labels_pred)

In [ ]:
def estimate_quality(metric, uniqueness_method, include_box_sections):
    pmids = []
    levels = []
    scores = []
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        pmid, _ = os.path.splitext(filename)
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
        analyzer = reload_exported_analyzer(analysis_file)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                         include_box_sections=include_box_sections)
            ground_truth = unique_ids_clustering(processed_clustering, method=uniqueness_method)
            score, _ = evaluate_clustering(analyzer, ground_truth, metric)
            print(pmid, level, score)
            pmids.append(pmid)
            levels.append(level)
            scores.append(score)
            
    return pmids, levels, scores 

In [ ]:
COLOR_MAP = {1: '#455cf5', 2: '#33e836', 3: '#eb5021'}

In [ ]:
scores_by_metric = {}

with PdfPages('pubtrends-quality.pdf') as pdf:
    for metric in [adjusted_rand_score, adjusted_mutual_info_score, v_measure_score, fowlkes_mallows_score]:
        for uniqueness_method in ['first_occurrence', 'unique_only']:
            for include_box_sections in [True, False]:
                print(metric.__name__, uniqueness_method, include_box_sections)
                pmids, levels, scores = estimate_quality(metric, uniqueness_method, include_box_sections)
                
                # Visualize
                colors = [COLOR_MAP[l] for l in levels]
    
                plt.figure(figsize=(10, 7))
                plt.scatter(pmids, scores, c=colors)
                plt.title(f'unique: {uniqueness_method} - include_box_sections: {include_box_sections}')
                plt.xlabel('PMID')
                plt.ylabel(metric.__name__)
                plt.xticks(rotation = 90)
                pdf.savefig()
                plt.close()
                
                if metric.__name__ not in scores_by_metric:
                    scores_by_metric[metric.__name__] = scores
                else:
                    scores_by_metric[metric.__name__].append(scores)

In [ ]:
import pandas as pd
from pandas.plotting import scatter_matrix

In [ ]:
scores_df = pd.DataFrame(scores_by_metric).apply(pd.to_numeric, errors='coerce')

In [ ]:
scatter_matrix(scores_df, alpha=0.8, grid=True, figsize=(15, 15))

In [ ]:
scores_df.head()

### Visualize Contingency Matrices for Default PubTrends Settings

In [ ]:
METRIC_ALIAS = {
    'adjusted_rand_score': 'Rand',
    'adjusted_mutual_info_score': 'Mutual-Info',
    'v_measure_score': 'V-Measure',
    'fowlkes_mallows_score': 'Fowlkes-Mallows'
}

with PdfPages('pubtrends-contingency-matrices.pdf') as pdf:
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        pmid, _ = os.path.splitext(filename)
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
        analyzer = reload_exported_analyzer(analysis_file)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                         include_box_sections=False)
            ground_truth = unique_ids_clustering(processed_clustering, method='unique_only')
            
            scores = {}
            for metric in [adjusted_rand_score, adjusted_mutual_info_score, 
                           v_measure_score, fowlkes_mallows_score]:
                scores[metric.__name__], cm = evaluate_clustering(analyzer, ground_truth, metric)
            
            metrics_str = ", ".join([f'{METRIC_ALIAS[k]}: {v:.2f}' for k, v in scores.items()])
            plt.figure(figsize = (10,7))
            sns.heatmap(cm, annot=True)
            plt.title(f'{filename} - LEVEL {level}\n{metrics_str}')
            plt.xlabel('PubTrends')
            plt.ylabel('Nature Reviews')
            pdf.savefig()
            plt.close()

### Topic Analysis Only for Subgraph (Direct Reference Papers, One of the Subtopics, etc)

In [ ]:
def get_direct_references_subgraph(analyzer, pmid):
    references = list(analyzer.citations_graph.successors(pmid))
    references.append(pmid)
    
    references_similarity_graph = analyzer.similarity_graph.subgraph(references)
    return references_similarity_graph

In [ ]:
from collections import Counter

def get_most_references_subtopic_subgraph(analyzer, pmid):
    references = list(analyzer.citations_graph.successors(pmid))
    reference_comps = analyzer.df[analyzer.df['id'].isin(references)]['comp'].values
    subtopic_id = Counter(reference_comps).most_common(1)[0][0]
    subtopic_papers = analyzer.df[analyzer.df['comp'] == subtopic_id]['id'].values
    
    subtopic_subgraph = analyzer.similarity_graph.subgraph(subtopic_papers)
    return subtopic_subgraph

In [ ]:
def evaluate_clustering_for_partition(partition, ground_truth, pmid, metric):
    # Get clustering
    actual_clustering = {k: v for k, v in partition.items() if k in ground_truth}
    
    # Align clusterings
    labels_true = []
    labels_pred = []

    for pmid in actual_clustering:
        labels_true.append(ground_truth[pmid])
        labels_pred.append(actual_clustering[pmid])
        
    return metric(labels_true, labels_pred), contingency_matrix(labels_true, labels_pred)

In [ ]:
def recalculate_topic_analysis(analyzer, graph=None, settings=AnalyzerSettings()):
    if not graph:
        graph = analyzer.similarity_graph
    topics_dendrogram, partition, comp_other, components, comp_sizes = \
                analyzer.topic_analysis(graph,
                                        topic_min_size=settings.TOPIC_MIN_SIZE,
                                        max_topics_number=settings.TOPICS_MAX_NUMBER,
                                        random_state=settings.SEED,
                                        similarity_bibliographic_coupling=settings.SIMILARITY_BIBLIOGRAPHIC_COUPLING,
                                        similarity_cocitation=settings.SIMILARITY_COCITATION,
                                        similarity_citation=settings.SIMILARITY_CITATION,
                                        similarity_text_citation=settings.SIMILARITY_TEXT_CITATION)
    return partition

#### Direct References

In [ ]:
METRIC_ALIAS = {
    'adjusted_rand_score': 'Rand',
    'adjusted_mutual_info_score': 'Mutual-Info',
    'v_measure_score': 'V-Measure',
    'fowlkes_mallows_score': 'Fowlkes-Mallows'
}

with PdfPages('pubtrends-contingency-matrices-refs-only.pdf') as pdf:
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        pmid, _ = os.path.splitext(filename)
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
        analyzer = reload_exported_analyzer(analysis_file)
        direct_references_subgraph = get_direct_references_subgraph(analyzer, pmid)
        partition = recalculate_topic_analysis(analyzer, direct_references_subgraph)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                         include_box_sections=False)
            ground_truth = unique_ids_clustering(processed_clustering, method='unique_only')
            
            scores = {}
            for metric in [adjusted_rand_score, adjusted_mutual_info_score, 
                           v_measure_score, fowlkes_mallows_score]:
                scores[metric.__name__], cm = evaluate_clustering_for_partition(partition, 
                                                                                ground_truth, 
                                                                                pmid, metric)
            
            metrics_str = ", ".join([f'{METRIC_ALIAS[k]}: {v:.2f}' for k, v in scores.items()])
            plt.figure(figsize = (10,7))
            sns.heatmap(cm, annot=True)
            plt.title(f'{filename} - LEVEL {level}\n{metrics_str}')
            plt.xlabel('PubTrends')
            plt.ylabel('Nature Reviews')
            pdf.savefig()
            plt.close()

#### The Biggest Subtopic

In [ ]:
METRIC_ALIAS = {
    'adjusted_rand_score': 'Rand',
    'adjusted_mutual_info_score': 'Mutual-Info',
    'v_measure_score': 'V-Measure',
    'fowlkes_mallows_score': 'Fowlkes-Mallows'
}

with PdfPages('pubtrends-contingency-matrices-biggest-subtopic.pdf') as pdf:
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        pmid, _ = os.path.splitext(filename)
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
        analyzer = reload_exported_analyzer(analysis_file)
        subtopic_subgraph = get_most_references_subtopic_subgraph(analyzer, pmid)
        if not subtopic_subgraph:
            print('Bad subgraph')
        partition = recalculate_topic_analysis(analyzer, subtopic_subgraph)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                         include_box_sections=False)
            ground_truth = unique_ids_clustering(processed_clustering, method='unique_only')
            
            scores = {}
            for metric in [adjusted_rand_score, adjusted_mutual_info_score, 
                           v_measure_score, fowlkes_mallows_score]:
                scores[metric.__name__], cm = evaluate_clustering_for_partition(partition, 
                                                                                ground_truth, 
                                                                                pmid, metric)
            
            metrics_str = ", ".join([f'{METRIC_ALIAS[k]}: {v:.2f}' for k, v in scores.items()])
            plt.figure(figsize = (10,7))
            sns.heatmap(cm, annot=True)
            plt.title(f'{filename} - LEVEL {level}\n{metrics_str}')
            plt.xlabel('PubTrends')
            plt.ylabel('Nature Reviews')
            pdf.savefig()
            plt.close()

### PubTrends Cluster Sizes

TODO: compare with only-refs setting

In [ ]:
from collections import Counter

for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
    pmid, _ = os.path.splitext(filename)
    full_filename = os.path.join(CLUSTERING_FOLDER, filename)
    with open(full_filename, 'r') as f:
        raw_clustering = json.load(f)
    analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
    analyzer = reload_exported_analyzer(analysis_file)
    processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                 include_box_sections=False)
    ground_truth = unique_ids_clustering(processed_clustering, method='unique_only')
    actual_clustering = get_clustering(analyzer, ground_truth.keys()).values()
    comp_sizes = Counter(actual_clustering).most_common(3)
    n_papers = len(actual_clustering)
    scaled_sizes = [v * 100 / n_papers for comp, v in comp_sizes]
    print(f"{pmid}\t{scaled_sizes}")

## Run Grid Search for Analyzer Parameters

In [ ]:
import itertools
import logging

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
param_grid = {
    'SIMILARITY_BIBLIOGRAPHIC_COUPLING': [0.125, 0.25, 0.5, 1, 2, 4, 8],
#     'SIMILARITY_COCITATION': [0.125, 0.25, 0.5, 1, 2, 4, 8],
#     'SIMILARITY_CITATION': [0.125, 0.25, 0.5, 1, 2, 4, 8],
    'SIMILARITY_TEXT_CITATION': [0.125, 0.25, 0.5, 1, 2, 4, 8]
}

In [ ]:
def settings_of_interest(analyzer_settings):
    return (
        analyzer_settings.SIMILARITY_BIBLIOGRAPHIC_COUPLING,
        analyzer_settings.SIMILARITY_COCITATION,
#         analyzer_settings.SIMILARITY_CITATION,
        analyzer_settings.SIMILARITY_TEXT_CITATION,
    )

In [ ]:
baseline_params = dict(
    TOPIC_MIN_SIZE=0,
    TOPICS_MAX_NUMBER=500,
    SIMILARITY_CITATION=0,
    SIMILARITY_COCITATION=1
)

In [ ]:
TARGET_METRIC = v_measure_score.__name__

In [ ]:
param_names = param_grid.keys()
results = {}

for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
    pmid, _ = os.path.splitext(filename)
    full_filename = os.path.join(CLUSTERING_FOLDER, filename)
    with open(full_filename, 'r') as f:
        raw_clustering = json.load(f)
    analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
    analyzer = reload_exported_analyzer(analysis_file)
    logging.info(f'{pmid} - loaded clustering and analyzer')
    rebuild_similarity_graph(analyzer)
    direct_references_subgraph = get_direct_references_subgraph(analyzer, pmid)
    logging.info(f'{pmid} - rebuilt similarity graph with scaling')
    
    for level in range(1, get_clustering_level(raw_clustering)):
        processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                     include_box_sections=False)
        ground_truth = unique_ids_clustering(processed_clustering, method='unique_only')
        logging.info(f'{pmid} level {level} - calculated ground truth clustering')
        
        best_score = 0
        best_soi = None
        grid_results = {}
        
        logging.info(f'{pmid} level {level} - starting grid search')
    
        # Iterate over param grid
        for param_values in itertools.product(*param_grid.values()):
            params = {k: v for k, v in zip(param_names, param_values)}
            for k, v in baseline_params.items():
                params[k] = v

            # Apply settings and recalculate 
            settings = AnalyzerSettings(**params)
            soi = settings_of_interest(settings)
            partition = recalculate_topic_analysis(analyzer, 
                                                   graph=direct_references_subgraph,
                                                   settings=settings)

            result = {}
            for metric in [adjusted_rand_score, adjusted_mutual_info_score, 
                           v_measure_score, fowlkes_mallows_score]:
                result[metric.__name__], _ = evaluate_clustering_for_partition(partition, 
                                                                               ground_truth, 
                                                                               pmid, metric)
            result['partition'] = partition
            grid_results[soi] = result
            
            n_comps = len(set(partition.values()))
            new_best = False
            if result[TARGET_METRIC] > best_score:
                best_score = result[TARGET_METRIC]
                best_soi = soi
                new_best = True

            score_str = f"{'BEST' if new_best else '    '} {result[TARGET_METRIC]}" 
            print(soi, score_str, n_comps, sep='\t')
            
        logging.info(f'{pmid} level {level} - best score {best_score} for params {best_soi}')
        best_partition = grid_results[best_soi]['partition']
        _, cm = evaluate_clustering_for_partition(partition, ground_truth, pmid, metric)
        logging.info('Best partition')
        print(cm)
        
        results[(pmid, level)] = {
            'best_score': best_score,
            'best_soi': best_soi,
            'grid': grid_results
        }

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
from datetime import datetime

In [ ]:
def now_str():
    return datetime.now().strftime('%Y_%m_%d_%H_%M')

In [ ]:
results_df.to_csv(f'grid-search_no-citations_refs-only_{now_str()}.csv')

In [ ]:
results_df_restored = pd.read_csv('grid-search_no-citations_refs-only_2021_02_10_02_31.csv',
                                  header=[0, 1], index_col=0).transpose()

In [ ]:
results_df_restored.best_score.idxmax()

In [ ]:
results_df_restored

In [ ]:
results_df_restored.best_score = results_df_restored.best_score.apply(float)

In [ ]:
results_df_restored.best_score.idxmax()

In [ ]:
def restore_soi(soi_str):
    return tuple([float(x) for x in soi_str.strip('()').split(',')])

In [ ]:
optim_diffs = {}
optim_bibcoupling = []
optim_text_citation = []

with PdfPages('pubtrends-contingency-matrices-scaled-grid-bibcoupling-text-citation.pdf') as pdf:
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        pmid, _ = os.path.splitext(filename)
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
        analyzer = reload_exported_analyzer(analysis_file)
        direct_references_subgraph = get_direct_references_subgraph(analyzer, pmid)
        partition = recalculate_topic_analysis(analyzer, direct_references_subgraph)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                         include_box_sections=False)
            ground_truth = unique_ids_clustering(processed_clustering, method='unique_only')
            
            pubtrends_score, _ = evaluate_clustering_for_partition(partition, 
                                                                   ground_truth, 
                                                                   pmid, v_measure_score)
            file_results = results_df_restored.loc[(pmid, str(level)), :]    
            optim_score = file_results['best_score']
            optim_soi = restore_soi(file_results['best_soi'])
            bc, _, tc = optim_soi
            optim_bibcoupling.append(bc)
            optim_text_citation.append(tc)
#             grid = json.loads(file_results['grid'])
            optim_diffs[(pmid, level)] = optim_score - pubtrends_score
            print(pubtrends_score, optim_score, optim_soi)

In [ ]:
plt.hist(optim_diffs.values()

In [ ]:
max_diff = max(optim_diffs.values())
for k, v in optim_diffs.items():
    if v == max_diff:
        print(k, v)

In [ ]:
import math

plt.hist2d([math.log2(x) for x in optim_bibcoupling], 
           [math.log2(x) for x in optim_text_citation],
           bins=(7, 7))
plt.xlabel('Bibcoupling')
plt.ylabel('Text Citation')
plt.colorbar()

In [ ]:
results_df

## Understanding Text Similarity

In [ ]:
analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'28003656.json.gz')
analyzer = reload_exported_analyzer(analysis_file)

In [ ]:
PUB_DF_COLUMNS = ['id', 'title', 'abstract', 'year', 'type', 'keywords', 'mesh', 'doi', 'aux']
analyzer.pub_df = analyzer.df[PUB_DF_COLUMNS]

In [ ]:
settings = AnalyzerSettings()
analyzer.corpus_ngrams, analyzer.corpus_counts = \
    vectorize_corpus(analyzer.pub_df,
                     max_features=settings.VECTOR_WORDS, n_gram=settings.VECTOR_NGRAMS,
                     min_df=settings.VECTOR_MIN_DF, max_df=settings.VECTOR_MAX_DF)

In [ ]:
tfidf = compute_tfidf(analyzer.corpus_counts)

In [ ]:
analyzer.corpus_counts

In [ ]:
tfidf.shape

In [ ]:
texts_similarity = analyzer.analyze_texts_similarity(analyzer.pub_df, tfidf,
                                                     settings.SIMILARITY_TEXT_MIN,
                                                     settings.SIMILARITY_TEXT_CITATION_N)

In [ ]:
text_citations = {}

pids = list(analyzer.df['id'])
for i, pid1 in enumerate(analyzer.df['id']):
    elements = []
    while not texts_similarity[i].empty():
        similarity, j = texts_similarity[i].get()
        elements.append((similarity, pids[j]))
    text_citations[pid1] = elements

In [ ]:
pid_ids = {pid: i for i, pid in enumerate(pids)}
for pid1, cit_list in text_citations.items():
    for el in cit_list:
        similarity, pid2 = el
        id1 = pid_ids[pid1]
        id2 = pid_ids[pid2]
        tfidf1 = tfidf[id1, :]
        tfidf2 = tfidf[id2, :]
        common_indices = set(tfidf1.indices).intersection(tfidf2.indices)
        n_words = tfidf.shape[1]
        print(similarity, pid, pid2)
        tfidf1 = tfidf1.toarray()
        tfidf2 = tfidf2.toarray()
        common_words = []
        for i in common_indices:
            w = tfidf1[0, i] * tfidf2[0, i]
            common_words.append((analyzer.corpus_ngrams[i], w))
        print(sorted(common_words, key=lambda el: el[1], reverse=True))
        print()

In [ ]:
import scipy
help(scipy.sparse.csr.csr_matrix)